# Traffic Demand Prediction - Advanced Modeling and Submission

This notebook implements our final improved pipeline, featuring:
1. **Decoded Latitude/Longitude Coordinates** from geohashes.
2. **Day-Lag & rolling window features** (1h & 2h rolling mean/std) from Day 48 with cascade fallback mechanisms.
3. **Spatial Hierarchical Morning Ratio Fallback** (geohash -> geo_5 -> geo_4 -> geo_3 -> global) to align Day 48 lag features to Day 49 morning levels.
4. **Day 49 morning trend features** using the first 2 hours of Day 49.
5. **Tuned Ensemble** of LightGBM, XGBoost, and HistGradientBoosting Regressors.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'dataset').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'dataset'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/underxcore/Desktop/Flipkart_GridLock


In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
print(f'Train shape: {train.shape}, Test shape: {test.shape}')

Train shape: (77299, 11), Test shape: (41778, 10)


In [3]:
def parse_time(df):
    out = df.copy()
    hm = out['timestamp'].astype(str).str.split(':', expand=True).astype(int)
    out['hour'] = hm[0]
    out['minute'] = hm[1]
    out['minute_of_day'] = out['hour'] * 60 + out['minute']
    out['global_minute'] = out['day'] * 24 * 60 + out['minute_of_day']
    out['time_sin'] = np.sin(2 * np.pi * out['minute_of_day'] / (24 * 60))
    out['time_cos'] = np.cos(2 * np.pi * out['minute_of_day'] / (24 * 60))
    out['is_morning_peak'] = out['hour'].between(7, 10).astype(int)
    out['is_evening_peak'] = out['hour'].between(16, 20).astype(int)
    out['is_night'] = ((out['hour'] <= 5) | (out['hour'] >= 22)).astype(int)
    return out

train = parse_time(train)
test = parse_time(test)

def decode_geohash(geohash):
    base32 = '0123456789bcdefghjkmnpqrstuvwxyz'
    lat_interval = (-90.0, 90.0)
    lon_interval = (-180.0, 180.0)
    is_even = True
    for char in geohash:
        val = base32.find(char)
        if val == -1:
            continue
        for i in range(4, -1, -1):
            bit = (val >> i) & 1
            if is_even:
                mid = (lon_interval[0] + lon_interval[1]) / 2
                if bit == 1:
                    lon_interval = (mid, lon_interval[1])
                else:
                    lon_interval = (lon_interval[0], mid)
            else:
                mid = (lat_interval[0] + lat_interval[1]) / 2
                if bit == 1:
                    lat_interval = (mid, lat_interval[1])
                else:
                    lat_interval = (lat_interval[0], mid)
            is_even = not is_even
    lat = (lat_interval[0] + lat_interval[1]) / 2
    lon = (lon_interval[0] + lon_interval[1]) / 2
    return lat, lon

print("Decoding geohashes...")
train[['lat', 'lon']] = train['geohash'].apply(lambda x: pd.Series(decode_geohash(x)))
test[['lat', 'lon']] = test['geohash'].apply(lambda x: pd.Series(decode_geohash(x)))

for df in [train, test]:
    df['geo_5'] = df['geohash'].astype(str).str[:5]
    df['geo_4'] = df['geohash'].astype(str).str[:4]
    df['geo_3'] = df['geohash'].astype(str).str[:3]
    df['geo_2'] = df['geohash'].astype(str).str[:2]
    df['geohash_len'] = df['geohash'].astype(str).str.len()
print('Time and geohash coordinates/prefix features added.')

Decoding geohashes...


Time and geohash coordinates/prefix features added.


In [4]:
print("Computing day 48 demand profiles...")
train_48 = train[train['day'] == 48]

df_pivot_48 = train_48.pivot_table(index='geohash', columns='minute_of_day', values='demand', aggfunc='mean')
df_pivot_48_geo_5 = train_48.pivot_table(index='geo_5', columns='minute_of_day', values='demand', aggfunc='mean')
df_pivot_48_geo_4 = train_48.pivot_table(index='geo_4', columns='minute_of_day', values='demand', aggfunc='mean')
df_pivot_48_geo_3 = train_48.pivot_table(index='geo_3', columns='minute_of_day', values='demand', aggfunc='mean')
df_pivot_48_geo_2 = train_48.pivot_table(index='geo_2', columns='minute_of_day', values='demand', aggfunc='mean')
df_pivot_48_global = train_48.pivot_table(columns='minute_of_day', values='demand', aggfunc='mean')

# Compute Rolling Profiles on Day 48
df_roll_mean_1h = df_pivot_48.T.rolling(window=5, center=True, min_periods=1).mean().T
df_roll_std_1h = df_pivot_48.T.rolling(window=5, center=True, min_periods=1).std().T
df_roll_mean_2h = df_pivot_48.T.rolling(window=9, center=True, min_periods=1).mean().T
df_roll_std_2h = df_pivot_48.T.rolling(window=9, center=True, min_periods=1).std().T

df_geo_5_mean_1h = df_pivot_48_geo_5.T.rolling(window=5, center=True, min_periods=1).mean().T
df_geo_5_std_1h = df_pivot_48_geo_5.T.rolling(window=5, center=True, min_periods=1).std().T
df_geo_5_mean_2h = df_pivot_48_geo_5.T.rolling(window=9, center=True, min_periods=1).mean().T
df_geo_5_std_2h = df_pivot_48_geo_5.T.rolling(window=9, center=True, min_periods=1).std().T

df_geo_4_mean_1h = df_pivot_48_geo_4.T.rolling(window=5, center=True, min_periods=1).mean().T
df_geo_4_std_1h = df_pivot_48_geo_4.T.rolling(window=5, center=True, min_periods=1).std().T
df_geo_4_mean_2h = df_pivot_48_geo_4.T.rolling(window=9, center=True, min_periods=1).mean().T
df_geo_4_std_2h = df_pivot_48_geo_4.T.rolling(window=9, center=True, min_periods=1).std().T

df_geo_3_mean_1h = df_pivot_48_geo_3.T.rolling(window=5, center=True, min_periods=1).mean().T
df_geo_3_std_1h = df_pivot_48_geo_3.T.rolling(window=5, center=True, min_periods=1).std().T
df_geo_3_mean_2h = df_pivot_48_geo_3.T.rolling(window=9, center=True, min_periods=1).mean().T
df_geo_3_std_2h = df_pivot_48_geo_3.T.rolling(window=9, center=True, min_periods=1).std().T

df_geo_2_mean_1h = df_pivot_48_geo_2.T.rolling(window=5, center=True, min_periods=1).mean().T
df_geo_2_std_1h = df_pivot_48_geo_2.T.rolling(window=5, center=True, min_periods=1).std().T
df_geo_2_mean_2h = df_pivot_48_geo_2.T.rolling(window=9, center=True, min_periods=1).mean().T
df_geo_2_std_2h = df_pivot_48_geo_2.T.rolling(window=9, center=True, min_periods=1).std().T

df_global_mean_1h = df_pivot_48_global.T.rolling(window=5, center=True, min_periods=1).mean().T
df_global_std_1h = df_pivot_48_global.T.rolling(window=5, center=True, min_periods=1).std().T
df_global_mean_2h = df_pivot_48_global.T.rolling(window=9, center=True, min_periods=1).mean().T
df_global_std_2h = df_pivot_48_global.T.rolling(window=9, center=True, min_periods=1).std().T

offsets = [-60, -45, -30, -15, 0, 15, 30, 45, 60]

def add_day_48_features(df):
    out = df.copy()
    for offset in offsets:
        df_offset = df_pivot_48.stack().reset_index()
        df_offset.columns = ['geohash', 'minute_of_day_target', f'demand_day_48_offset_{offset}']
        df_offset['minute_of_day'] = df_offset['minute_of_day_target'] - offset
        out = out.merge(df_offset[['geohash', 'minute_of_day', f'demand_day_48_offset_{offset}']], 
                        on=['geohash', 'minute_of_day'], how='left')
        
        df_offset_geo_5 = df_pivot_48_geo_5.stack().reset_index()
        df_offset_geo_5.columns = ['geo_5', 'minute_of_day_target', f'demand_day_48_geo_5_offset_{offset}']
        df_offset_geo_5['minute_of_day'] = df_offset_geo_5['minute_of_day_target'] - offset
        out = out.merge(df_offset_geo_5[['geo_5', 'minute_of_day', f'demand_day_48_geo_5_offset_{offset}']], 
                        on=['geo_5', 'minute_of_day'], how='left')
        
        df_offset_geo_4 = df_pivot_48_geo_4.stack().reset_index()
        df_offset_geo_4.columns = ['geo_4', 'minute_of_day_target', f'demand_day_48_geo_4_offset_{offset}']
        df_offset_geo_4['minute_of_day'] = df_offset_geo_4['minute_of_day_target'] - offset
        out = out.merge(df_offset_geo_4[['geo_4', 'minute_of_day', f'demand_day_48_geo_4_offset_{offset}']], 
                        on=['geo_4', 'minute_of_day'], how='left')
        
        df_offset_geo_3 = df_pivot_48_geo_3.stack().reset_index()
        df_offset_geo_3.columns = ['geo_3', 'minute_of_day_target', f'demand_day_48_geo_3_offset_{offset}']
        df_offset_geo_3['minute_of_day'] = df_offset_geo_3['minute_of_day_target'] - offset
        out = out.merge(df_offset_geo_3[['geo_3', 'minute_of_day', f'demand_day_48_geo_3_offset_{offset}']], 
                        on=['geo_3', 'minute_of_day'], how='left')

        df_offset_geo_2 = df_pivot_48_geo_2.stack().reset_index()
        df_offset_geo_2.columns = ['geo_2', 'minute_of_day_target', f'demand_day_48_geo_2_offset_{offset}']
        df_offset_geo_2['minute_of_day'] = df_offset_geo_2['minute_of_day_target'] - offset
        out = out.merge(df_offset_geo_2[['geo_2', 'minute_of_day', f'demand_day_48_geo_2_offset_{offset}']], 
                        on=['geo_2', 'minute_of_day'], how='left')

        df_offset_global = df_pivot_48_global.stack().reset_index()
        df_offset_global.columns = ['dummy', 'minute_of_day_target', f'demand_day_48_global_offset_{offset}']
        df_offset_global['minute_of_day'] = df_offset_global['minute_of_day_target'] - offset
        out = out.merge(df_offset_global[['minute_of_day', f'demand_day_48_global_offset_{offset}']], 
                        on=['minute_of_day'], how='left')
        
        col_geohash = f'demand_day_48_offset_{offset}'
        col_geo_5 = f'demand_day_48_geo_5_offset_{offset}'
        col_geo_4 = f'demand_day_48_geo_4_offset_{offset}'
        col_geo_3 = f'demand_day_48_geo_3_offset_{offset}'
        col_geo_2 = f'demand_day_48_geo_2_offset_{offset}'
        col_global = f'demand_day_48_global_offset_{offset}'
        
        out[col_geohash] = (out[col_geohash]
                            .fillna(out[col_geo_5])
                            .fillna(out[col_geo_4])
                            .fillna(out[col_geo_3])
                            .fillna(out[col_geo_2])
                            .fillna(out[col_global]))
        
        out.drop(columns=[col_geo_5, col_geo_4, col_geo_3, col_geo_2, col_global], inplace=True)
        out.loc[out['day'] == 48, col_geohash] = np.nan

    # Rolling window stats
    rolling_profiles = {
        'roll_mean_1h': (df_roll_mean_1h, df_geo_5_mean_1h, df_geo_4_mean_1h, df_geo_3_mean_1h, df_geo_2_mean_1h, df_global_mean_1h),
        'roll_std_1h': (df_roll_std_1h, df_geo_5_std_1h, df_geo_4_std_1h, df_geo_3_std_1h, df_geo_2_std_1h, df_global_std_1h),
        'roll_mean_2h': (df_roll_mean_2h, df_geo_5_mean_2h, df_geo_4_mean_2h, df_geo_3_mean_2h, df_geo_2_mean_2h, df_global_mean_2h),
        'roll_std_2h': (df_roll_std_2h, df_geo_5_std_2h, df_geo_4_std_2h, df_geo_3_std_2h, df_geo_2_std_2h, df_global_std_2h),
    }
    
    for name, profiles in rolling_profiles.items():
        p_gh, p_g5, p_g4, p_g3, p_g2, p_gb = profiles
        
        df_f = p_gh.stack().reset_index()
        df_f.columns = ['geohash', 'minute_of_day', f'demand_day_48_{name}']
        out = out.merge(df_f, on=['geohash', 'minute_of_day'], how='left')
        
        df_f_g5 = p_g5.stack().reset_index()
        df_f_g5.columns = ['geo_5', 'minute_of_day', f'demand_day_48_geo_5_{name}']
        out = out.merge(df_f_g5, on=['geo_5', 'minute_of_day'], how='left')
        
        df_f_g4 = p_g4.stack().reset_index()
        df_f_g4.columns = ['geo_4', 'minute_of_day', f'demand_day_48_geo_4_{name}']
        out = out.merge(df_f_g4, on=['geo_4', 'minute_of_day'], how='left')
        
        df_f_g3 = p_g3.stack().reset_index()
        df_f_g3.columns = ['geo_3', 'minute_of_day', f'demand_day_48_geo_3_{name}']
        out = out.merge(df_f_g3, on=['geo_3', 'minute_of_day'], how='left')

        df_f_g2 = p_g2.stack().reset_index()
        df_f_g2.columns = ['geo_2', 'minute_of_day', f'demand_day_48_geo_2_{name}']
        out = out.merge(df_f_g2, on=['geo_2', 'minute_of_day'], how='left')

        df_f_gb = p_gb.stack().reset_index()
        df_f_gb.columns = ['dummy', 'minute_of_day', f'demand_day_48_global_{name}']
        out = out.merge(df_f_gb[['minute_of_day', f'demand_day_48_global_{name}']], on=['minute_of_day'], how='left')
        
        col_gh = f'demand_day_48_{name}'
        col_g5 = f'demand_day_48_geo_5_{name}'
        col_g4 = f'demand_day_48_geo_4_{name}'
        col_g3 = f'demand_day_48_geo_3_{name}'
        col_g2 = f'demand_day_48_geo_2_{name}'
        col_gb = f'demand_day_48_global_{name}'
        
        out[col_gh] = (out[col_gh]
                       .fillna(out[col_g5])
                       .fillna(out[col_g4])
                       .fillna(out[col_g3])
                       .fillna(out[col_g2])
                       .fillna(out[col_gb]))
        
        out.drop(columns=[col_g5, col_g4, col_g3, col_g2, col_gb], inplace=True)
        out.loc[out['day'] == 48, col_gh] = np.nan
    return out

print("Merging profiles...")
train_fe = add_day_48_features(train)
test_fe = add_day_48_features(test)

Computing day 48 demand profiles...
Merging profiles...


In [5]:
print("Computing day 49 morning averages...")
train_49_morning = train[(train['day'] == 49) & (train['minute_of_day'] <= 120)]
day_49_morning_avg = train_49_morning.groupby('geohash')['demand'].mean().to_frame('demand_day_49_morning_avg').reset_index()
train_fe = train_fe.merge(day_49_morning_avg, on='geohash', how='left')
test_fe = test_fe.merge(day_49_morning_avg, on='geohash', how='left')

train_48_morning = train[(train['day'] == 48) & (train['minute_of_day'] <= 120)]
day_48_morning_avg = train_48_morning.groupby('geohash')['demand'].mean().to_frame('demand_day_48_morning_avg').reset_index()

print("Computing hierarchical ratios...")
ratio_gh = day_49_morning_avg.merge(day_48_morning_avg, on='geohash', how='inner')
ratio_gh['ratio_gh'] = ratio_gh['demand_day_49_morning_avg'] / (ratio_gh['demand_day_48_morning_avg'] + 1e-5)
ratio_gh['ratio_gh'] = ratio_gh['ratio_gh'].clip(0.5, 3.5)

train_49_geo_5 = train_49_morning.groupby('geo_5')['demand'].mean().to_frame('demand_day_49_morning_avg_geo_5').reset_index()
train_48_geo_5 = train_48[train_48['minute_of_day'] <= 120].groupby('geo_5')['demand'].mean().to_frame('demand_day_48_morning_avg_geo_5').reset_index()
ratio_g5 = train_49_geo_5.merge(train_48_geo_5, on='geo_5', how='inner')
ratio_g5['ratio_g5'] = ratio_g5['demand_day_49_morning_avg_geo_5'] / (ratio_g5['demand_day_48_morning_avg_geo_5'] + 1e-5)
ratio_g5['ratio_g5'] = ratio_g5['ratio_g5'].clip(0.5, 3.5)

train_49_geo_4 = train_49_morning.groupby('geo_4')['demand'].mean().to_frame('demand_day_49_morning_avg_geo_4').reset_index()
train_48_geo_4 = train_48[train_48['minute_of_day'] <= 120].groupby('geo_4')['demand'].mean().to_frame('demand_day_48_morning_avg_geo_4').reset_index()
ratio_g4 = train_49_geo_4.merge(train_48_geo_4, on='geo_4', how='inner')
ratio_g4['ratio_g4'] = ratio_g4['demand_day_49_morning_avg_geo_4'] / (ratio_g4['demand_day_48_morning_avg_geo_4'] + 1e-5)
ratio_g4['ratio_g4'] = ratio_g4['ratio_g4'].clip(0.5, 3.5)

train_49_geo_3 = train_49_morning.groupby('geo_3')['demand'].mean().to_frame('demand_day_49_morning_avg_geo_3').reset_index()
train_48_geo_3 = train_48[train_48['minute_of_day'] <= 120].groupby('geo_3')['demand'].mean().to_frame('demand_day_48_morning_avg_geo_3').reset_index()
ratio_g3 = train_49_geo_3.merge(train_48_geo_3, on='geo_3', how='inner')
ratio_g3['ratio_g3'] = ratio_g3['demand_day_49_morning_avg_geo_3'] / (ratio_g3['demand_day_48_morning_avg_geo_3'] + 1e-5)
ratio_g3['ratio_g3'] = ratio_g3['ratio_g3'].clip(0.5, 3.5)

global_ratio = 1.556785

train_fe = train_fe.merge(ratio_gh[['geohash', 'ratio_gh']], on='geohash', how='left')
train_fe = train_fe.merge(ratio_g5[['geo_5', 'ratio_g5']], on='geo_5', how='left')
train_fe = train_fe.merge(ratio_g4[['geo_4', 'ratio_g4']], on='geo_4', how='left')
train_fe = train_fe.merge(ratio_g3[['geo_3', 'ratio_g3']], on='geo_3', how='left')
train_fe['ratio'] = (train_fe['ratio_gh']
                     .fillna(train_fe['ratio_g5'])
                     .fillna(train_fe['ratio_g4'])
                     .fillna(train_fe['ratio_g3'])
                     .fillna(global_ratio))
train_fe.drop(columns=['ratio_gh', 'ratio_g5', 'ratio_g4', 'ratio_g3'], inplace=True)

test_fe = test_fe.merge(ratio_gh[['geohash', 'ratio_gh']], on='geohash', how='left')
test_fe = test_fe.merge(ratio_g5[['geo_5', 'ratio_g5']], on='geo_5', how='left')
test_fe = test_fe.merge(ratio_g4[['geo_4', 'ratio_g4']], on='geo_4', how='left')
test_fe = test_fe.merge(ratio_g3[['geo_3', 'ratio_g3']], on='geo_3', how='left')
test_fe['ratio'] = (test_fe['ratio_gh']
                    .fillna(test_fe['ratio_g5'])
                    .fillna(test_fe['ratio_g4'])
                    .fillna(test_fe['ratio_g3'])
                    .fillna(global_ratio))
test_fe.drop(columns=['ratio_gh', 'ratio_g5', 'ratio_g4', 'ratio_g3'], inplace=True)

day_48_morning_avg_fallback = day_48_morning_avg.rename(columns={'demand_day_48_morning_avg': 'demand_day_49_morning_avg_fallback'})
train_fe = train_fe.merge(day_48_morning_avg_fallback, on='geohash', how='left')
train_fe['demand_day_49_morning_avg'] = train_fe['demand_day_49_morning_avg'].fillna(train_fe['demand_day_49_morning_avg_fallback'])
train_fe.drop(columns=['demand_day_49_morning_avg_fallback'], inplace=True)

test_fe = test_fe.merge(day_48_morning_avg_fallback, on='geohash', how='left')
test_fe['demand_day_49_morning_avg'] = test_fe['demand_day_49_morning_avg'].fillna(test_fe['demand_day_49_morning_avg_fallback'])
test_fe.drop(columns=['demand_day_49_morning_avg_fallback'], inplace=True)

for offset in offsets:
    col = f'demand_day_48_offset_{offset}'
    train_fe[f'{col}_scaled'] = train_fe[col] * train_fe['ratio']
    test_fe[f'{col}_scaled'] = test_fe[col] * test_fe['ratio']

day_49_latest = train_49_morning.sort_values('minute_of_day').groupby('geohash').last().reset_index()
day_49_latest_val = day_49_latest[['geohash', 'demand', 'minute_of_day']].rename(
    columns={'demand': 'demand_day_49_latest', 'minute_of_day': 'minute_day_49_latest'}
)
train_fe = train_fe.merge(day_49_latest_val, on='geohash', how='left')
test_fe = test_fe.merge(day_49_latest_val, on='geohash', how='left')

train_fe['demand_day_49_latest_diff'] = train_fe['minute_of_day'] - train_fe['minute_day_49_latest']
test_fe['demand_day_49_latest_diff'] = test_fe['minute_of_day'] - test_fe['minute_day_49_latest']
train_fe.drop(columns=['minute_day_49_latest'], inplace=True)
test_fe.drop(columns=['minute_day_49_latest'], inplace=True)

train_fe['Temperature_missing'] = train_fe['Temperature'].isna().astype(int)
train_fe['RoadType_missing'] = train_fe['RoadType'].isna().astype(int)
train_fe['Weather_missing'] = train_fe['Weather'].isna().astype(int)

test_fe['Temperature_missing'] = test_fe['Temperature'].isna().astype(int)
test_fe['RoadType_missing'] = test_fe['RoadType'].isna().astype(int)
test_fe['Weather_missing'] = test_fe['Weather'].isna().astype(int)
print('Morning average and latest observed features added.')

Computing day 49 morning averages...
Computing hierarchical ratios...
Morning average and latest observed features added.


In [6]:
target = 'demand'
exclude = ['Index', target, 'timestamp']
feature_cols = [c for c in train_fe.columns if c not in exclude]

categorical_features = [
    'geohash', 'geo_2', 'geo_3', 'geo_4', 'geo_5',
    'RoadType', 'LargeVehicles', 'Landmarks', 'Weather',
]
numeric_features = [c for c in feature_cols if c not in categorical_features]

ordered = train_fe.sort_values('global_minute')
cutoff = ordered['global_minute'].quantile(0.80)
train_mask = train_fe['global_minute'] <= cutoff

X_train = train_fe.loc[train_mask, feature_cols]
y_train = train_fe.loc[train_mask, target]
X_valid = train_fe.loc[~train_mask, feature_cols]
y_valid = train_fe.loc[~train_mask, target]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=5, sparse_output=False)),
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ],
    remainder='drop',
)

X_train_proc = preprocessor.fit_transform(X_train)
X_valid_proc = preprocessor.transform(X_valid)

print('Evaluating Models on Validation Split...')
m_lgbm = LGBMRegressor(n_estimators=500, learning_rate=0.04, max_depth=6, num_leaves=31, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1)
m_lgbm.fit(X_train_proc, y_train)
p_lgbm = np.clip(m_lgbm.predict(X_valid_proc), 0, 1)

m_xgb = XGBRegressor(n_estimators=700, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1)
m_xgb.fit(X_train_proc, y_train)
p_xgb = np.clip(m_xgb.predict(X_valid_proc), 0, 1)

m_hgb = HistGradientBoostingRegressor(loss='squared_error', learning_rate=0.03, max_iter=600, max_leaf_nodes=31, l2_regularization=0.03, random_state=42)
m_hgb.fit(X_train_proc, y_train)
p_hgb = np.clip(m_hgb.predict(X_valid_proc), 0, 1)

p_ens = 0.70 * p_lgbm + 0.20 * p_xgb + 0.10 * p_hgb
print(f'LightGBM Validation R2: {r2_score(y_valid, p_lgbm):.5f}')
print(f'XGBoost Validation R2: {r2_score(y_valid, p_xgb):.5f}')
print(f'HGB Validation R2: {r2_score(y_valid, p_hgb):.5f}')
print(f'Ensemble Validation R2: {r2_score(y_valid, p_ens):.5f}')

Evaluating Models on Validation Split...


LightGBM Validation R2: 0.72877
XGBoost Validation R2: 0.73051
HGB Validation R2: 0.71950
Ensemble Validation R2: 0.72976


In [7]:
print('Training final models on all available data...')
X_train_full = train_fe[feature_cols]
y_train_full = train_fe[target]
X_test_full = test_fe[feature_cols]

X_train_full_proc = preprocessor.fit_transform(X_train_full)
X_test_full_proc = preprocessor.transform(X_test_full)

m_lgbm.fit(X_train_full_proc, y_train_full)
f_lgbm = np.clip(m_lgbm.predict(X_test_full_proc), 0, 1)

m_xgb.fit(X_train_full_proc, y_train_full)
f_xgb = np.clip(m_xgb.predict(X_test_full_proc), 0, 1)

m_hgb.fit(X_train_full_proc, y_train_full)
f_hgb = np.clip(m_hgb.predict(X_test_full_proc), 0, 1)

final_predictions = np.clip(0.70 * f_lgbm + 0.20 * f_xgb + 0.10 * f_hgb, 0, 1)

submission = pd.DataFrame({
    'Index': test['Index'],
    'demand': final_predictions
})

submission.to_csv('submission.csv', index=False)
print('Saved submission to submission.csv')
print(submission['demand'].describe())

Training final models on all available data...


Saved submission to submission.csv
count    41778.000000
mean         0.129805
std          0.173236
min          0.001463
25%          0.029708
50%          0.060263
75%          0.142482
max          1.000000
Name: demand, dtype: float64
